# WTI Producer Hedge Simulator

**Question:** How much can a crude producer reduce revenue risk by hedging expected production with WTI futures?

The producer owns future oil production, so falling oil prices can reduce revenue. This notebook tests how short WTI futures can help reduce that risk.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Download and prepare WTI data

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare different hedge sizes for 100,000 barrels/month

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: what happens if crude prices fall 25%?

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Midland vs. Cushing basis risk

A WTI futures hedge can reduce the main oil-price risk, but Midland and Cushing prices may still move differently. That remaining price difference is basis risk.

The examples below are simple stress-test scenarios, not a historical Midland cash-price backtest.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### What this means

Even if the main WTI price is fully hedged, a wider Midland discount can still reduce revenue. A separate basis hedge can help offset that difference.

## Find the hedge size that reduced risk the most

This section uses historical spot and futures price changes to estimate the hedge ratio that reduced price risk the most.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
